# Business Forecasting — Chapter 2: Time-Series Graphics
## DETAILED SELF-STUDY VERSION

Use this notebook after class. It covers the same lesson as the live version, but explains the Python syntax and forecasting logic in much greater detail.

**Study method:** Read the explanation above a code cell, read the comments inside the code, run the cell, then interpret the output before moving on.


# 1. Setup: the Business Forecasting Python toolkit

**Learning goals**
- Understand what a Python library is.
- Learn the standard imports we will reuse throughout Business Forecasting.
- Learn why we use short names such as `pd`, `np`, `plt`, and `sns`.

## Detailed explanation
### What every line means

`import pandas as pd` loads **pandas** and creates the short alias `pd`. An alias is simply a nickname. It changes nothing about the library itself.

`import numpy as np` does the same for NumPy.

`import matplotlib.pyplot as plt` loads the plotting part of Matplotlib. Most introductory Python data-analysis courses use the nickname `plt`.

`import seaborn as sns` loads Seaborn. Seaborn works on top of Matplotlib and is useful for plots such as scatterplots and grouped line charts.

`import statsmodels.api as sm` gives us access to statistical functions. Later, `sm.tsa.acf(...)` will calculate autocorrelations.

`from statsmodels.graphics.tsaplots import plot_acf` imports one specific plotting function directly. Because we import the function itself, we can write `plot_acf(...)` instead of a longer library path.

`sns.set_theme(...)` changes appearance only. It does not change the data or analysis.

`pd.set_option(...)` changes how pandas displays values.

**Business Forecasting habit:** Keep one standard setup cell near the top of every forecasting notebook.

### What you should be able to explain after this section
**Python becomes useful for forecasting because libraries provide reliable data-handling, graphing, and statistical tools.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Business Forecasting STANDARD SETUP — you will use these libraries often in this course.

import pandas as pd                    # pandas: tables, CSV files, dates, filtering, grouping
import numpy as np                     # NumPy: numerical calculations and random numbers
import matplotlib.pyplot as plt        # Matplotlib: basic plotting
import seaborn as sns                  # Seaborn: convenient statistical graphics
import statsmodels.api as sm           # statsmodels: statistical/time-series tools
from statsmodels.graphics.tsaplots import plot_acf  # ACF graph used repeatedly in forecasting

sns.set_theme(style="whitegrid")       # Give our graphs a clean, readable background
pd.set_option("display.precision", 3)  # Show most decimal numbers to 3 places

print("Business Forecasting setup complete.")

# 2. DataFrames: how Python stores business data

A **DataFrame** is a two-dimensional table. Rows are observations; columns are variables. Before using a real dataset, we will build a tiny one ourselves.

## Detailed explanation
A Python **dictionary** uses `{}` and stores pairs of labels and values.

Here, `"Year"` is a dictionary key and `[2019, ...]` is its associated list of values.

`pd.DataFrame(example_data)` constructs a DataFrame. The result has:
- 5 rows;
- 2 columns;
- an automatically created row index from 0 to 4.

The final line contains only `sales_df`. In a Jupyter/Colab notebook, writing the name of an object as the final line displays it automatically.

**Business Forecasting habit:** Before modelling, always know what one row represents and what every column represents.

### What you should be able to explain after this section
**A DataFrame is the standard table structure we will use for most Business Forecasting work.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Create a Python dictionary. Each key will become a column name.
example_data = {
    "Year": [2019, 2020, 2021, 2022, 2023],
    "Sales": [120, 135, 128, 155, 171]
}

# Convert the dictionary into a pandas DataFrame (a table).
sales_df = pd.DataFrame(example_data)

# Display the whole small table.
sales_df

# 3. Inspecting a DataFrame before analysis

Never begin forecasting by blindly graphing or modelling. First inspect the structure, data types, missing values, and a few rows.

## Detailed explanation
`shape` is an **attribute**, so there are no parentheses after it. It returns a tuple such as `(5, 2)`.

`columns` is also an attribute. `.tolist()` converts pandas' Index object into an ordinary list for easier reading.

`dtypes` reports each column's data type.

`.head()` is a **method**, so it uses parentheses. By default it returns the first five rows.

`.describe()` summarizes numerical columns. It is useful for detecting surprising scales or values, but it does **not** replace visual inspection.

**Business Forecasting standard workflow:** inspect → clean → visualize → model → diagnose → forecast.

### What you should be able to explain after this section
**Inspection comes before forecasting.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
print("Shape:", sales_df.shape)            # (number of rows, number of columns)
print("Columns:", sales_df.columns.tolist()) # Convert column labels to a normal Python list
print("\nData types:")
print(sales_df.dtypes)                     # Show the data type stored in each column

print("\nFirst rows:")
display(sales_df.head())                   # Show the first 5 rows

print("\nSummary:")
display(sales_df.describe())               # Numerical summary: count, mean, std, min, quartiles, max

# 4. Dates and time stamps

Forecasting data are ordered in time. Python must understand a date as a date, not merely as text. We will learn `pd.to_datetime()` and the `.dt` date accessor.

## Detailed explanation
`pd.Series(...)` creates a one-dimensional pandas object.

`pd.to_datetime(dates)` parses recognizable date strings and returns datetime values.

When a **Series** contains datetime values, date-specific operations are accessed through `.dt`.

`monthly_df["Month"].dt.year` extracts the year as an integer.

`monthly_df["Month"].dt.month_name()` returns readable month names.

**Important:** `.dt` works only when pandas recognizes the column as a datetime-like type.

**Business Forecasting habit:** after loading a dataset, check the date column's `dtype`.

### What you should be able to explain after this section
**Time-series analysis requires correctly parsed dates.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Start with date values written as text.
dates = pd.Series(["2024-01-01", "2024-02-01", "2024-03-01"])

# Convert the text into true pandas datetime values.
dates = pd.to_datetime(dates)

# Build a small monthly time-series DataFrame.
monthly_df = pd.DataFrame({
    "Month": dates,
    "Demand": [210, 225, 238]
})

# Extract useful pieces of the date using the .dt accessor.
monthly_df["Year"] = monthly_df["Month"].dt.year
monthly_df["MonthName"] = monthly_df["Month"].dt.month_name()

monthly_df

# 5. Loading real Chapter 2 data from a CSV

We now move to real data. `pd.read_csv()` reads a comma-separated file directly into a DataFrame. `parse_dates=` tells pandas which column should be interpreted as a date while loading.

## Detailed explanation
`pbs_url = "..."` stores a URL in a variable. This is useful because we can reuse the address without retyping it.

`pd.read_csv(pbs_url, parse_dates=["Month"])` downloads and parses the CSV into a DataFrame. In Google Colab this requires internet access.

`pbs[[...]]` selects several columns. The **outer** brackets mean “select from the DataFrame”; the **inner list** contains the names of the columns to keep.

`pbs.info()` reports row count, column names, non-null counts, and data types.

`display(...)` is provided by Jupyter/Colab and renders DataFrames neatly.

**Business Forecasting habit:** use `parse_dates=` whenever you know which CSV column contains time.

### What you should be able to explain after this section
**Load data carefully and inspect it immediately.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# URL of the Pharmaceutical Benefits Scheme dataset used in Chapter 2.
pbs_url = "https://otexts.com/fpppy/data/PBS_unparsed.csv"

# Read the CSV file directly from the book website.
# parse_dates=["Month"] tells pandas to convert Month to datetime immediately.
pbs = pd.read_csv(pbs_url, parse_dates=["Month"])

# Keep the columns we need for this lesson.
pbs = pbs[["Month", "Concession", "Type", "ATC1", "ATC2", "Scripts", "Cost"]]

# Check the result before doing analysis.
print(pbs.info())
display(pbs.head())

# 6. Filtering rows with `.loc[]`

Filtering means keeping only observations that meet a condition. Here we select pharmaceutical records whose ATC2 code is `A10`.

## Detailed explanation
`pbs["ATC2"] == "A10"` performs an element-by-element comparison.

The result is a Boolean Series containing `True` and `False`.

`pbs.loc[is_a10]` uses that Boolean Series as a row filter.

`.copy()` creates an independent copy of the filtered result and helps avoid confusing chained-assignment behaviour.

`len(a10)` returns the number of rows.

**Python syntax to remember**
- `=` means assignment: put a value into a variable.
- `==` means comparison: ask whether two values are equal.

**Business Forecasting habit:** isolate the exact series you intend to analyse before forecasting it.

### What you should be able to explain after this section
**Filtering turns a large dataset into the specific business series you want to study.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Create a Boolean condition: True where ATC2 equals "A10", False elsewhere.
is_a10 = pbs["ATC2"] == "A10"

# .loc[condition] keeps only rows where the condition is True.
a10 = pbs.loc[is_a10].copy()

# Display a sample of the filtered data.
display(a10.head())

# Check how many rows remain.
print("Rows after filtering:", len(a10))

# 7. Aggregation with `groupby()`

The A10 data contain several records per month because of different concession and payment categories. To obtain one monthly cost series, we group by month and sum Cost.

## Detailed explanation
A **groupby-aggregate** operation has two conceptual steps:

1. **Split** rows into groups using one or more variables.
2. **Aggregate** values within each group using a function such as sum, mean, min, or max.

`a10.groupby("Month", as_index=False)` creates monthly groups.

`.agg({"Cost": "sum"})` tells pandas to sum `Cost` inside each group.

`1_000_000` is simply one million. Python allows underscores inside large numbers to improve readability.

**Forecasting reason:** a single time series normally needs one target value for each time point. Duplicate time points must be understood and, when appropriate, aggregated.

**Business Forecasting habit:** always ask, “Do I have one observation per time period for this series?”

### What you should be able to explain after this section
**`groupby()` is how we turn detailed transactional records into a usable time series.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Group all A10 observations that belong to the same Month.
# as_index=False keeps Month as a normal column.
total_cost_df = (
    a10.groupby("Month", as_index=False)
       .agg({"Cost": "sum"})
)

# Convert dollars to millions of dollars for easier graph labels.
total_cost_df["CostMillions"] = total_cost_df["Cost"] / 1_000_000

# Keep only the two columns needed for the time series.
total_cost_df = total_cost_df[["Month", "CostMillions"]]

display(total_cost_df.head())

# 8. Seasonal period: the number you must identify

The **seasonal period** is the number of observations required for a seasonal pattern to repeat.

Common examples:
- quarterly data → 4 observations per year;
- monthly data → 12;
- weekly data → about 52;
- daily data with weekly seasonality → 7;
- hourly data with daily seasonality → 24.

## Detailed explanation
The seasonal period is often written as **m** in forecasting texts.

Examples:
- Monthly + annual repetition: `m = 12`.
- Quarterly + annual repetition: `m = 4`.
- Daily + weekly repetition: `m = 7`.
- Half-hourly + daily repetition: `m = 48`.

A dataset can have **multiple seasonal periods**. Half-hourly electricity demand can repeat within days, weeks, and years.

**Business Forecasting habit:** identify both the **data frequency** and the likely **seasonal period** before choosing a forecasting method.

### What you should be able to explain after this section
**Frequency tells how often we observe; seasonal period tells how often a repeating pattern returns.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
seasonal_period_examples = pd.DataFrame({
    "Data frequency": ["Quarterly", "Monthly", "Weekly", "Daily", "Hourly"],
    "Common seasonal period": [4, 12, 52, 7, 24],
    "Pattern represented": ["Year", "Year", "Year", "Week", "Day"]
})

seasonal_period_examples

# 9. Time plots: the first graph in forecasting

A **time plot** places time on the horizontal axis and the observed value on the vertical axis. Consecutive observations are connected so we can see structure over time.

## Detailed explanation
`plt.subplots(...)` creates a **Figure** and an **Axes** object.
- `fig` represents the whole figure.
- `ax` represents the plotting area.

`ax.plot(x, y)` creates a line plot.

`ax.set(...)` sets multiple labels at once.

`plt.show()` explicitly displays the finished figure.

When reading a time plot, look for:
1. **trend** — long-run increase or decrease;
2. **seasonality** — repeating pattern tied to calendar frequency;
3. **cycles** — rises and falls without a fixed period;
4. **outliers/unusual observations**;
5. **changes in variability**.

**Business Forecasting rule:** plot the time series before choosing a model.

### What you should be able to explain after this section
**Graphs are diagnostic tools, not decoration.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Create a plotting area.
fig, ax = plt.subplots(figsize=(10, 4))

# Draw CostMillions against Month.
ax.plot(total_cost_df["Month"], total_cost_df["CostMillions"])

# Add labels that explain exactly what the viewer is seeing.
ax.set(
    title="Australian antidiabetic drug sales",
    xlabel="Month",
    ylabel="Cost (millions of Australian dollars)"
)

# Display the graph.
plt.show()

# 10. Trend, seasonality, and cycles

Three terms must be distinguished:
- **Trend:** long-term movement in the level.
- **Seasonality:** a repeating pattern with a known, fixed calendar frequency.
- **Cycle:** rises and falls without a fixed frequency.

## Detailed explanation
`.copy()` prevents changes to `season_df` from modifying the original `total_cost_df`.

`.dt.year` extracts the calendar year.

`.dt.month` extracts month number 1–12.

`.dt.strftime("%b")` formats each date as a three-letter month abbreviation such as Jan, Feb, Mar.

These helper variables will allow us to compare the same season across multiple years.

**Business Forecasting habit:** date feature extraction is common. You will repeatedly see `.dt.year`, `.dt.month`, `.dt.quarter`, and `.dt.day_name()`.

### What you should be able to explain after this section
**Dates contain useful structure that can be extracted for analysis.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Add year and month information so we can examine the monthly seasonal pattern.
season_df = total_cost_df.copy()
season_df["Year"] = season_df["Month"].dt.year
season_df["MonthNumber"] = season_df["Month"].dt.month
season_df["MonthName"] = season_df["Month"].dt.strftime("%b")

# Preview the new columns.
display(season_df.head())

# 11. Seasonal plot

A seasonal plot overlays each year's values against the months of the year. This makes the within-year seasonal pattern much easier to see.

## Detailed explanation
`sns.lineplot(...)` can split a dataset into multiple lines using the `hue=` argument.

`hue="Year"` means Seaborn creates a separate line for each year.

`sort=False` tells Seaborn not to alphabetically rearrange the month labels.

`ax.set_xticks(range(12), month_order)` explicitly labels positions 0–11 with calendar month abbreviations.

`bbox_to_anchor` moves the legend outside the plot.

**Interpretation:** Seasonality is supported when the within-year shape recurs across years.

**Business Forecasting habit:** when seasonality matters, do not rely on a time plot alone; use a seasonal view too.

### What you should be able to explain after this section
**A seasonal plot compares the same positions within each seasonal cycle.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Define the correct month order so the x-axis runs Jan to Dec.
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Create the graph.
fig, ax = plt.subplots(figsize=(10, 5))

# Draw one line for each year.
sns.lineplot(
    data=season_df,
    x="MonthName",
    y="CostMillions",
    hue="Year",
    sort=False,
    ax=ax
)

# Force calendar month order and add labels.
ax.set_xticks(range(12), month_order)
ax.set(
    title="Seasonal plot: antidiabetic drug sales",
    xlabel="Month of year",
    ylabel="Cost (millions)"
)

# Move the legend outside so it does not cover the data.
ax.legend(title="Year", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.show()

# 12. Seasonal subseries plot

A **seasonal subseries plot** groups all January observations together, all February observations together, and so on. It helps compare seasonal levels across time.

## Detailed explanation
`plt.subplots(3, 4, ...)` creates 12 plotting panels.

`.flatten()` turns the 3×4 array of axes into a simple one-dimensional sequence, making it easier to loop over.

`for month_number in range(1, 13):` repeats the indented code for month numbers 1 through 12. Python's `range(1, 13)` stops before 13.

`.loc[...]` filters the DataFrame to one month.

`marker="o"` displays each observation with a circular marker.

`.mean()` computes the arithmetic average.

`ax.axhline(...)` adds a horizontal reference line.

**Python concept:** indentation is part of Python syntax. Every line belonging to the `for` loop must be indented consistently.

### What you should be able to explain after this section
**Subseries plots separate seasons so their levels and changes can be compared directly.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Create a 3-by-4 grid: one small graph for each month.
fig, axes = plt.subplots(3, 4, figsize=(12, 8), sharey=True)
axes = axes.flatten()

# Loop through months 1 to 12.
for month_number in range(1, 13):
    ax = axes[month_number - 1]

    # Keep only observations belonging to the current month.
    one_month = season_df.loc[season_df["MonthNumber"] == month_number]

    # Plot that month's values across years.
    ax.plot(one_month["Year"], one_month["CostMillions"], marker="o")

    # Add a horizontal line showing the average for this month.
    ax.axhline(one_month["CostMillions"].mean(), linestyle="--")

    # Add the month name as the panel title.
    ax.set_title(month_order[month_number - 1])

fig.suptitle("Seasonal subseries plot: antidiabetic drug sales")
fig.supxlabel("Year")
fig.supylabel("Cost (millions)")
plt.tight_layout()
plt.show()

# 13. Multiple seasonal periods: electricity demand

Some data have more than one repeating seasonal pattern. Half-hourly electricity demand can have daily, weekly, and annual patterns.

## Detailed explanation
The CSV follows a forecasting-friendly format:
- `unique_id` identifies the series;
- `ds` stores the date/time;
- `y` stores the observed value.

The filter keeps only rows where `unique_id == "Demand"`.

The second filter uses two conditions joined by `&`.
Each condition must be inside parentheses.

For half-hourly observations:
- daily seasonal period = `48`;
- weekly seasonal period = `48 * 7 = 336`.

**Business Forecasting concept:** higher-frequency data often contain multiple seasonalities.

### What you should be able to explain after this section
**A time series can repeat at more than one frequency.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Load Victorian half-hourly electricity data from Chapter 2.
vic_url = "https://otexts.com/fpppy/data/vic_elec.csv"
vic_elec = pd.read_csv(vic_url, parse_dates=["ds"])

# Keep only the Demand series.
demand = vic_elec.loc[vic_elec["unique_id"] == "Demand"].copy()

# Use only one week to make the classroom graph easy to read.
one_week = demand.loc[
    (demand["ds"] >= "2014-01-06") &
    (demand["ds"] <  "2014-01-13")
].copy()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(one_week["ds"], one_week["y"])
ax.set(
    title="Victorian electricity demand — one week",
    xlabel="Date and time",
    ylabel="Demand"
)
plt.xticks(rotation=45)
plt.show()

# 14. Scatterplots: relationships between variables

Forecasting sometimes uses predictor variables. A scatterplot helps us see the relationship between two numerical variables.
Here we compare electricity demand with temperature.

## Detailed explanation
`.pivot(index="ds", columns="unique_id", values="y")` changes the data from **long** format to **wide** format.

Before pivoting, Demand and Temperature are identified by values in `unique_id`.
After pivoting, they become separate columns aligned by time.

`reset_index()` turns `ds` back into a regular column.

`sns.scatterplot(...)` draws one point for every paired Temperature–Demand observation.

**Forecasting lesson:** always visualize relationships before relying on one summary statistic.

### What you should be able to explain after this section
**Scatterplots reveal relationships that a correlation number can hide.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Reshape the data so Demand and Temperature become separate columns.
elec_wide = (
    vic_elec.pivot(index="ds", columns="unique_id", values="y")
            .reset_index()
)

# Keep observations from 2014.
elec_2014 = elec_wide.loc[elec_wide["ds"].dt.year == 2014].copy()

# Draw Temperature on x and Demand on y.
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=elec_2014,
    x="Temperature",
    y="Demand",
    s=12,
    ax=ax
)
ax.set(
    title="Electricity demand versus temperature",
    xlabel="Temperature",
    ylabel="Electricity demand"
)
plt.show()

# 15. Correlation — useful, but not enough

The correlation coefficient measures the strength and direction of a **linear** relationship. It ranges from -1 to +1.

## Detailed explanation
`.dropna()` removes rows containing missing values so both variables are paired consistently.

`.corr(...)` computes the Pearson correlation by default.

`f"...{r:.3f}"` is an f-string. It inserts the value of `r` into the text and formats it to three decimal places.

Interpretation:
- `r ≈ +1`: strong positive linear association;
- `r ≈ 0`: weak linear association;
- `r ≈ -1`: strong negative linear association.

A value near zero does **not** prove that the variables are unrelated; the relationship may be nonlinear.

**Business Forecasting habit:** graph first, statistic second.

### What you should be able to explain after this section
**Correlation summarizes linear association only.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Keep only rows where both variables are present.
corr_data = elec_2014[["Temperature", "Demand"]].dropna()

# Compute the Pearson correlation coefficient.
r = corr_data["Temperature"].corr(corr_data["Demand"])

print(f"Correlation between Temperature and Demand: {r:.3f}")

# 16. Lag: comparing a series with its own past

A **lag** moves a time series backward by a specified number of observations. `shift(1)` creates the previous observation, `shift(4)` creates the observation four periods earlier, and so on.

## Detailed explanation
`.shift(k)` moves values down by `k` rows so each current observation is aligned with a value from `k` periods earlier.

For quarterly data:
- lag 1 = previous quarter;
- lag 4 = same quarter one year earlier;
- lag 8 = same quarter two years earlier.

The missing values created at the beginning are mathematically unavoidable — no earlier observation exists inside the available sample.

**Business Forecasting concept:** lags connect current behaviour to historical behaviour and are fundamental in autocorrelation, ARIMA, and regression with lagged predictors.

### What you should be able to explain after this section
**A lag is a previous value aligned with the current observation.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Load quarterly Australian beer production.
beer_url = "https://otexts.com/fpppy/data/aus_production.csv"
beer = pd.read_csv(beer_url, parse_dates=["ds"])

# Keep only dates from 2000 onward and only the columns we need.
beer = beer.loc[beer["ds"] >= "2000", ["ds", "Beer"]].copy()

# Rename Beer to y so the target has a short generic name.
beer = beer.rename(columns={"Beer": "y"})

# Create lagged versions of the target.
beer["lag_1"] = beer["y"].shift(1)  # one quarter earlier
beer["lag_4"] = beer["y"].shift(4)  # four quarters = one year earlier

display(beer.head(8))

# 17. Lag plots

A lag plot places the current value \(y_t\) against a lagged value \(y_{t-k}\). Repeating the plot for different lags can reveal seasonal dependence.

## Detailed explanation
`zip(range(1, 5), axes.flatten())` pairs lag numbers 1–4 with the four plotting panels.

`lagged = beer["y"].shift(lag)` creates a different lag each time the loop runs.

`ax.scatter(lagged, beer["y"])` plots past values on the x-axis and current values on the y-axis.

A seasonal series often shows stronger lag relationships at multiples of its seasonal period.

**Business Forecasting link:** lag plots are a visual bridge between raw time plots and the ACF.

### What you should be able to explain after this section
**Lag plots show how current values relate to their own past.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Create four lag plots for lags 1 through 4.
fig, axes = plt.subplots(2, 2, figsize=(8, 8))

for lag, ax in zip(range(1, 5), axes.flatten()):
    lagged = beer["y"].shift(lag)

    ax.scatter(lagged, beer["y"], s=20)
    ax.set(
        title=f"Lag {lag}",
        xlabel=f"y(t-{lag})",
        ylabel="y(t)"
    )

plt.tight_layout()
plt.show()

# 18. Autocorrelation and the ACF

**Autocorrelation** is correlation between a time series and its own lagged values. The **ACF** displays autocorrelation across many lags.

## Detailed explanation
`sm.tsa.acf(...)` calculates the autocorrelation function.

`nlags=12` requests lag 0 through lag 12.

`fft=False` uses the direct calculation. For this small teaching dataset, clarity matters more than computational speed.

`plot_acf(...)` draws the same concept graphically.

`zero=False` hides lag 0. Lag 0 is always exactly 1 because a series is perfectly correlated with itself.

The shaded confidence region gives a rough visual guide for judging which correlations are unusually far from zero.

**Business Forecasting habit:** interpret the ACF together with the time plot and the known data frequency.

### What you should be able to explain after this section
**The ACF summarizes dependence between present values and many past lags.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Calculate autocorrelations from lag 0 through lag 12.
acf_values = sm.tsa.acf(beer["y"].dropna(), nlags=12, fft=False)

# Put the values in a small table.
acf_table = pd.DataFrame({
    "Lag": range(len(acf_values)),
    "ACF": acf_values
})

display(acf_table)

# Plot the ACF.
fig, ax = plt.subplots(figsize=(9, 4))
plot_acf(
    beer["y"].dropna(),
    lags=16,
    zero=False,
    ax=ax
)
ax.set(
    title="ACF of quarterly beer production",
    xlabel="Lag (quarters)",
    ylabel="Autocorrelation"
)
plt.show()

# 19. Trend and seasonality in an ACF

A trended series often has large positive autocorrelations that decline slowly. A seasonal series often has larger ACF spikes at seasonal lags.

## Detailed explanation
For monthly data with annual seasonality, the seasonal period is 12.

Therefore seasonal ACF behaviour may appear at:
`12, 24, 36, 48, ...`

A slow ACF decay can be associated with trend or other forms of persistence.

**Important:** the ACF itself does not automatically tell you the final forecasting model. It is evidence that must be combined with plots, transformations, differencing, and later model diagnostics.

### What you should be able to explain after this section
**ACF patterns reflect structure already present in the time series.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Plot the ACF of monthly drug sales, which contains trend and seasonality.
fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(
    total_cost_df["CostMillions"].dropna(),
    lags=48,
    zero=False,
    ax=ax
)
ax.set(
    title="ACF of monthly antidiabetic drug sales",
    xlabel="Lag (months)",
    ylabel="Autocorrelation"
)
plt.show()

# 20. White noise: the forecasting benchmark

A **white-noise** series has no predictable autocorrelation structure. Its observations fluctuate randomly around a stable level with roughly constant variation.

## Detailed explanation
`np.random.default_rng(seed=30)` creates a modern NumPy random-number generator.

A fixed **seed** makes the random example reproducible. Every run produces the same sequence, which is useful in teaching and debugging.

`.normal(loc=0, scale=1, size=50)` creates 50 values from a normal distribution with mean 0 and standard deviation 1.

A white-noise process ideally has:
- no meaningful trend;
- no recurring seasonality;
- approximately zero autocorrelation at non-zero lags.

**Business Forecasting connection:** good forecasting residuals should often resemble white noise. This becomes an important model-diagnostic principle later.

### What you should be able to explain after this section
**If forecast errors still contain structure, the model has probably missed information.**


In [ ]:
# SELF-STUDY CODE — read the comments before running each line.
# Use a fixed random seed so everyone in class gets the same random numbers.
rng = np.random.default_rng(seed=30)

# Generate 50 random observations from a standard normal distribution.
white_noise = pd.Series(rng.normal(loc=0, scale=1, size=50), name="y")

# Plot the random series.
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(range(1, 51), white_noise)
ax.set(
    title="Example of white noise",
    xlabel="Observation",
    ylabel="Value"
)
plt.show()

# Plot its autocorrelation function.
fig, ax = plt.subplots(figsize=(9, 3))
plot_acf(white_noise, lags=16, zero=False, ax=ax)
ax.set(
    title="ACF of white noise",
    xlabel="Lag",
    ylabel="Autocorrelation"
)
plt.show()


# Class Work / Assignment — Chapter 2: Time-Series Graphics

## Purpose
This is not a coding-speed exercise. Your goal is to show that you can **inspect a time series, create appropriate graphics, and explain what the graphics mean for forecasting**.

### Dataset
Use the `tute1.csv` dataset from the FPP Python book:

`https://otexts.com/fpppy/data/tute1.csv`

It contains quarterly `Sales`, `AdBudget`, and `GDP` series.

## Part A — Data inspection
1. Import the dataset with `pd.read_csv()` and parse `ds` as a date.
2. Display the first 10 rows.
3. Use `.info()` and answer:
   - How many observations are present?
   - Which columns are numerical?
   - Is `ds` recognized as a date?
4. Identify the data frequency.
5. State the annual seasonal period for quarterly data.

## Part B — Time plots
6. Make a time plot of `Sales`.
7. Make a time plot of `AdBudget`.
8. Make a time plot of `GDP`.
9. For each graph, write 2–4 sentences discussing:
   - trend;
   - possible seasonality;
   - cycles;
   - unusual observations.

## Part C — Relationships
10. Make a scatterplot of `Sales` versus `AdBudget`.
11. Calculate their correlation.
12. Explain why the scatterplot should be examined **before** interpreting the correlation number.
13. Make a scatterplot of `Sales` versus `GDP`.
14. Which variable appears more useful as a possible predictor of Sales? Explain from the graphs, not only the correlation.

## Part D — Lag and ACF
15. Create `Sales_lag1` using `.shift(1)`.
16. Create `Sales_lag4` using `.shift(4)`.
17. Explain in words what lag 4 means for quarterly data.
18. Create a lag-1 scatterplot for Sales.
19. Create a lag-4 scatterplot for Sales.
20. Plot the ACF of Sales for at least 16 lags.
21. Is there evidence of seasonal dependence? Explain which lags support your answer.

## Part E — Forecasting interpretation
22. In one paragraph, describe the main time-series features of Sales.
23. What characteristics would a future forecasting model need to capture?
24. Why would fitting a model **without first plotting the series** be risky?
25. Write one question about the data that the graphs cannot answer by themselves.

### Submission
Submit your completed Colab notebook. Every graph must have a meaningful title and axis labels. Put your written answers in **Text/Markdown cells**, not as Python comments.

### Business Forecasting expectation
You are not graded only on whether Python runs. You are graded on whether you can explain **what the output means for forecasting**.


In [ ]:
# ASSIGNMENT STARTER CELL
# Complete the questions above in additional Code and Text cells.

tute1_url = "https://otexts.com/fpppy/data/tute1.csv"
tute1 = pd.read_csv(tute1_url, parse_dates=["ds"])

display(tute1.head(10))
tute1.info()